# 02. Dim 側 File Pruning 検証
「Dim を WHERE で絞ったとき、全ファイルスキャンされるのか？」に実データで回答する

<div style="background: linear-gradient(135deg, #FF3621 0%, #C0210F 100%); color: white; padding: 24px 32px; border-radius: 8px; margin-bottom: 24px;">
  <div style="font-size: 26px; font-weight: 700;">02. Dim 側 File Pruning 検証</div>
  <div style="font-size: 15px; opacity: 0.9; margin-top: 6px;">「Dim を WHERE で絞ったとき、全ファイルスキャンされるのか？」に実データで回答する</div>
</div>

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin-bottom: 12px;">
  <div style="font-weight: 700; color: #0d47a1; margin-bottom: 6px;">📌 お客様の質問</div>
  <div style="color: #1a237e; line-height: 1.7;">
    Fact ⋈ Dim の JOIN で <code>WHERE dim.prefecture = '東京都'</code> と絞ったとき、<br>
    Dim 側は <b>(a) 全レコードがスキャンされる</b> のか、<b>(b) 絞られたレコードだけがスキャンされる</b> のか？<br>
    背景: Dim テーブルの行数が多いため、Dim 側のスキャンコストが懸念されている。
  </div>
</div>

<div style="border-left: 4px solid #388E3C; background: #E8F5E9; padding: 16px 20px; border-radius: 4px;">
  <div style="font-weight: 700; color: #1B5E20; margin-bottom: 6px;">✅ 先に結論</div>
  <div style="color: #1B5E20; line-height: 1.7;">
    <b>Dim を <code>CLUSTER BY (prefecture)</code> で構成しておけば、東京都が入っているファイルだけが読まれ、それ以外は物理的にスキップされる</b>（File Pruning）。<br>
    全ファイルスキャンにはならない。ただし <b>CLUSTER BY で指定していないカラムで WHERE すると pruning は効かない</b>。
  </div>
</div>

## 🔑 EXPLAIN の見方（この3つだけ探せばOK）

<div style="background: #FAFAFA; border: 1px solid #E0E0E0; border-radius: 8px; padding: 20px; margin: 12px 0;">
  <table style="width: 100%; border-collapse: collapse; font-size: 13px;">
    <thead style="background: #263238; color: white;">
      <tr>
        <th style="padding: 10px; text-align: left; border: 1px solid #37474F;">見る場所</th>
        <th style="padding: 10px; text-align: left; border: 1px solid #37474F;">探すキーワード</th>
        <th style="padding: 10px; text-align: left; border: 1px solid #37474F;">意味</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">Dim テーブルの Scan ブロック（<code>PhotonScan ...dim_customer_...</code>）</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC; font-family: monospace;">DictionaryFilters: [(prefecture = 東京都)]</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">Dim 側で「東京都のデータが入っているファイルだけ読む」宣言</td>
      </tr>
      <tr style="background: #F5F5F5;">
        <td style="padding: 10px; border: 1px solid #CFD8DC;">ツリー真ん中</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC; font-family: monospace;">PhotonBroadcastHashJoin</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">小さい Dim を Broadcast して結合（DFP の前提）</td>
      </tr>
      <tr>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">Fact テーブルの Scan ブロック（<code>PhotonScan ...fact_sales</code>）</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC; font-family: monospace;">OptionalDataFilters: [hashedrelationcontains(...)]</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">Fact 側も Dim の絞り込み結果で pruning される（DFP 発火）</td>
      </tr>
    </tbody>
  </table>
</div>

<div style="border-left: 4px solid #F57C00; background: #FFF3E0; padding: 12px 18px; border-radius: 4px;">
  <div style="color: #BF360C; line-height: 1.7;">
    ⚠️ <b>実測「Files read」は Databricks の Query Profile で確認</b>してください。<br>
    SQL Editor / SQL Warehouse の「Query History」→ 対象クエリを開く → 「Query Profile」タブ → Scan ノードの「Files read」を見る。
  </div>
</div>

## 🧪 検証環境

In [0]:
USE CATALOG konomi_demo_catalog;
USE SCHEMA star_schema_pruning_demo;

-- 検証で使う3つの Dim テーブル
SELECT 'dim_customer_clustered' AS table_name, COUNT(*) AS row_count,
       'CLUSTER BY (prefecture)' AS clustering, '1億行、Prefecture でクラスタリング' AS note
FROM dim_customer_clustered
UNION ALL
SELECT 'dim_customer_unclustered', COUNT(*), 'なし', '1億行、クラスタリングなし（比較用）'
FROM dim_customer_unclustered
UNION ALL
SELECT 'fact_sales', COUNT(*), 'CLUSTER BY (customer_id)', '5億行、Fact テーブル'
FROM fact_sales
ORDER BY table_name;

In [0]:
-- Dim テーブルの詳細（clustered: 5ファイル、unclustered: 64ファイルの想定）
DESCRIBE DETAIL dim_customer_clustered;

In [0]:
DESCRIBE DETAIL dim_customer_unclustered;

In [0]:
-- Prefecture の分布（各県 10%）
SELECT prefecture, COUNT(*) AS customers,
       ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM dim_customer_clustered
GROUP BY prefecture
ORDER BY customers DESC;

## シナリオ A · Dim 単体スキャン（CLUSTER BY なし）
「もし何も工夫しなかったら」の基準値。ほぼ全ファイル読まれる。

<div style="background: linear-gradient(135deg, #6A1B9A 0%, #4A148C 100%); color: white; padding: 16px 24px; border-radius: 8px; margin: 12px 0;">
  <div style="font-size: 18px; font-weight: 700;">シナリオ A · Dim 単体スキャン（CLUSTER BY なし）</div>
  <div style="opacity: 0.9; margin-top: 2px; font-size: 13px;">対象: dim_customer_unclustered (64 ファイル) / 東京都のカウントだけ取得</div>
</div>

In [0]:
-- クエリ実行
SELECT COUNT(*) AS tokyo_customers
FROM dim_customer_unclustered
WHERE prefecture = '東京都';

In [0]:
-- EXPLAIN で pruning の宣言を確認
EXPLAIN FORMATTED
SELECT COUNT(*)
FROM dim_customer_unclustered
WHERE prefecture = '東京都';

<div style="background: #FAFAFA; border: 1px solid #E0E0E0; border-radius: 8px; padding: 20px; margin: 12px 0;">
  <div style="font-weight: 700; color: #263238; font-size: 15px; margin-bottom: 10px;">🔍 シナリオ A の読み解き</div>
  <div style="color: #37474F; line-height: 1.8; font-size: 14px;">
    <b>EXPLAIN の確認ポイント</b>:<br>
    ・<code>DictionaryFilters: [(prefecture = 東京都)]</code> は <b>あります</b>（Delta は pruning を試みる）<br>
    ・でも <b>クラスタリングされていない</b>ので、東京都のデータが全ファイルに散らばっている状態<br>
    ・結果: Query Profile を見ると <b>ほぼ 64ファイル全部読まれる</b><br><br>
    <b>💡 意味</b>: Delta は「pruning しようとしている」が、物理配置が悪いので実効効果が出ない。<br>
    <b>🎯 教訓</b>: CLUSTER BY を設定しないと、いくら Delta が賢くても File Pruning は効かない。
  </div>
</div>

## シナリオ B · Dim 単体スキャン（CLUSTER BY prefecture）
「ちゃんと設計した場合」。東京都が入ったファイルだけ読まれる。

<div style="background: linear-gradient(135deg, #6A1B9A 0%, #4A148C 100%); color: white; padding: 16px 24px; border-radius: 8px; margin: 12px 0;">
  <div style="font-size: 18px; font-weight: 700;">シナリオ B · Dim 単体スキャン（CLUSTER BY prefecture）</div>
  <div style="opacity: 0.9; margin-top: 2px; font-size: 13px;">対象: dim_customer_clustered (5 ファイル) / 東京都のカウントだけ取得</div>
</div>

In [0]:
-- クエリ実行（同じ結果 = 1000万人）
SELECT COUNT(*) AS tokyo_customers
FROM dim_customer_clustered
WHERE prefecture = '東京都';

In [0]:
-- EXPLAIN で pruning の宣言を確認
EXPLAIN FORMATTED
SELECT COUNT(*)
FROM dim_customer_clustered
WHERE prefecture = '東京都';

<div style="background: #FAFAFA; border: 1px solid #E0E0E0; border-radius: 8px; padding: 20px; margin: 12px 0;">
  <div style="font-weight: 700; color: #263238; font-size: 15px; margin-bottom: 10px;">🔍 シナリオ B の読み解き</div>
  <div style="color: #37474F; line-height: 1.8; font-size: 14px;">
    <b>EXPLAIN の確認ポイント</b>:<br>
    ・<code>DictionaryFilters: [(prefecture = 東京都)]</code> があり、かつ prefecture でクラスタリングされている<br>
    ・東京都のデータは <b>特定のファイルに集約</b>されている（今回は 5 ファイル中 1〜2 ファイルに集約されるはず）<br>
    ・結果: Query Profile で <b>「Files read」がグッと少なくなる</b><br><br>
    <b>💡 意味</b>: 東京都のデータが物理的にまとまっているので、それ以外のファイルは <b>1バイトも読まれない</b>。<br>
    <b>🎯 お客様の質問への直接の答え</b>: <b>「(b) 絞られたレコードだけスキャンされる」寄り。</b>正確には「東京都のデータが入っているファイルだけ物理的に読まれる」。
  </div>
</div>

## シナリオ C · Fact + Dim JOIN + 東京都フィルタ（実運用パターン）
実際のクエリで、Dim 側の pruning + Fact 側の DFP が同時に起きることを確認

<div style="background: linear-gradient(135deg, #6A1B9A 0%, #4A148C 100%); color: white; padding: 16px 24px; border-radius: 8px; margin: 12px 0;">
  <div style="font-size: 18px; font-weight: 700;">シナリオ C · Fact + Dim JOIN + 東京都フィルタ</div>
  <div style="opacity: 0.9; margin-top: 2px; font-size: 13px;">対象: fact_sales ⋈ dim_customer_clustered / 東京都の売上合計</div>
</div>

In [0]:
-- 実際のお客様の質問シナリオそのもの
SELECT COUNT(*) AS row_count, SUM(f.revenue) AS total_revenue
FROM fact_sales f
JOIN dim_customer_clustered c ON f.customer_id = c.customer_id
WHERE c.prefecture = '東京都';

In [0]:
-- EXPLAIN で Dim 側 pruning + Fact 側 DFP の両方を確認
EXPLAIN FORMATTED
SELECT COUNT(*), SUM(f.revenue)
FROM fact_sales f
JOIN dim_customer_clustered c ON f.customer_id = c.customer_id
WHERE c.prefecture = '東京都';

<div style="background: #FAFAFA; border: 1px solid #E0E0E0; border-radius: 8px; padding: 20px; margin: 12px 0;">
  <div style="font-weight: 700; color: #263238; font-size: 15px; margin-bottom: 10px;">🔍 シナリオ C の読み解き（EXPLAIN の 3 か所を確認）</div>
  <div style="color: #37474F; line-height: 1.8; font-size: 14px;">
    <b>① ツリー真ん中: <code>PhotonBroadcastHashJoin</code></b><br>
    &nbsp;&nbsp;→ 小さい Dim（絞った後）が Broadcast されている = DFP の前提条件クリア<br><br>
    <b>② Dim 側 Scan ブロック: <code>DictionaryFilters: [(prefecture = 東京都)]</code></b><br>
    &nbsp;&nbsp;→ Dim 側は東京都が入っているファイルだけ読む（お客様の質問への直接の答え）<br><br>
    <b>③ Fact 側 Scan ブロック: <code>OptionalDataFilters: [hashedrelationcontains(customer_id)]</code></b><br>
    &nbsp;&nbsp;→ Dim 側で絞った customer_id を使って Fact 側も pruning（DFP 発火）<br>
  </div>
</div>

<div style="background: #E8F5E9; border-left: 4px solid #388E3C; padding: 16px 20px; border-radius: 4px; margin: 12px 0;">
  <div style="font-weight: 700; color: #1B5E20; margin-bottom: 6px;">💡 結論（お客様への一言）</div>
  <div style="color: #1B5E20; line-height: 1.8; font-size: 14px;">
    「<b>Dim を CLUSTER BY (prefecture) で構成しておけば、東京都のデータが入っているファイルだけを物理的に読み込みます。それ以外のファイルは 1バイトも読まれません（File Pruning）</b>。<br>
    さらに、Databricks は <b>Dim 側の絞り込み結果を使って Fact 側の I/O も減らします（Dynamic File Pruning）</b>。<br>
    ただし、これらの pruning が効くには、<b>WHERE 句で使う列を CLUSTER BY で指定していることが前提</b>です。」
  </div>
</div>

## 📊 3シナリオの Query Profile 比較（実測値記入用）

<div style="background: #FAFAFA; border: 1px solid #E0E0E0; border-radius: 8px; padding: 20px; margin: 12px 0;">
  <div style="font-weight: 700; color: #263238; font-size: 15px; margin-bottom: 12px;">実際の Files read / Bytes read は Query Profile で確認してください</div>
  <table style="width: 100%; border-collapse: collapse; font-size: 13px;">
    <thead style="background: #263238; color: white;">
      <tr>
        <th style="padding: 10px; text-align: left; border: 1px solid #37474F;">シナリオ</th>
        <th style="padding: 10px; text-align: left; border: 1px solid #37474F;">対象</th>
        <th style="padding: 10px; text-align: left; border: 1px solid #37474F;">総ファイル数</th>
        <th style="padding: 10px; text-align: left; border: 1px solid #37474F;">Files read (実測)</th>
        <th style="padding: 10px; text-align: left; border: 1px solid #37474F;">Skip率</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">A: Dim (Unclustered)</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">dim_customer_unclustered</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">64</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">（Query Profile から記入）</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">（記入）</td>
      </tr>
      <tr style="background: #F5F5F5;">
        <td style="padding: 10px; border: 1px solid #CFD8DC;">B: Dim (Clustered)</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">dim_customer_clustered</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">5</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">（Query Profile から記入）</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">（記入）</td>
      </tr>
      <tr>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">C: Fact ⋈ Dim (Clustered)</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">fact_sales の pruning</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">64</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">（Query Profile から記入）</td>
        <td style="padding: 10px; border: 1px solid #CFD8DC;">（記入）</td>
      </tr>
    </tbody>
  </table>
</div>

## 🎯 総合まとめ

<div style="background: linear-gradient(135deg, #0B3D91 0%, #002171 100%); color: white; padding: 20px 28px; border-radius: 8px; margin: 12px 0;">
  <div style="font-size: 20px; font-weight: 700;">🎯 お客様の質問への回答（決定版）</div>
</div>

<table style="width: 100%; border-collapse: collapse; font-size: 14px; margin-top: 12px;">
  <thead style="background: #263238; color: white;">
    <tr>
      <th style="padding: 12px; text-align: left; border: 1px solid #37474F;">観点</th>
      <th style="padding: 12px; text-align: left; border: 1px solid #37474F;">結論</th>
      <th style="padding: 12px; text-align: left; border: 1px solid #37474F;">前提条件</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="padding: 12px; border: 1px solid #CFD8DC; font-weight: 600;">Dim 側スキャン</td>
      <td style="padding: 12px; border: 1px solid #CFD8DC;">対象値が入っている <b>ファイルだけ読まれる</b>（他のファイルは物理的にスキップ）</td>
      <td style="padding: 12px; border: 1px solid #CFD8DC;">Dim が <b>WHERE 句のカラムで CLUSTER BY されている</b>こと</td>
    </tr>
    <tr style="background: #F5F5F5;">
      <td style="padding: 12px; border: 1px solid #CFD8DC; font-weight: 600;">Fact 側スキャン</td>
      <td style="padding: 12px; border: 1px solid #CFD8DC;">Dim 側の絞り込み結果を使って <b>Fact 側の対象ファイルも減らされる</b>（Dynamic File Pruning）</td>
      <td style="padding: 12px; border: 1px solid #CFD8DC;"><b>Broadcast Join</b> になっていること + Fact が <b>JOIN キーで CLUSTER BY</b> されていること</td>
    </tr>
    <tr>
      <td style="padding: 12px; border: 1px solid #CFD8DC; font-weight: 600;">CLUSTER BY なしの Dim</td>
      <td style="padding: 12px; border: 1px solid #CFD8DC;">pruning を試みるが <b>実効的にはほぼ全ファイル読み</b></td>
    <td style="padding: 12px; border: 1px solid #CFD8DC;">設計時に CLUSTER BY を決めておくのが定石</td>
    </tr>
  </tbody>
</table>

## 📚 参考ドキュメント

<div style="background: #F5F5F5; border-radius: 8px; padding: 20px; margin: 12px 0;">
  <ul style="color: #37474F; line-height: 2; font-size: 14px;">
    <li><a href="https://docs.databricks.com/aws/ja/optimizations/dynamic-file-pruning">Databricks Docs: 動的ファイルプルーニング</a></li>
    <li><a href="https://docs.databricks.com/aws/ja/delta/clustering">Databricks Docs: Liquid Clustering</a></li>
    <li><a href="https://www.databricks.com/discover/pages/optimize-data-workloads-guide">Comprehensive Guide to Optimize Databricks Workloads</a></li>
    <li><a href="https://www.databricks.com/blog/2022/06/24/data-warehousing-modeling-techniques-and-their-implementation-on-the-databricks-lakehouse-platform.html">Data Warehousing Modeling Techniques on Databricks Lakehouse</a></li>
  </ul>
</div>